In [ ]:
# Cell 1: Import libraries and load the processed UK and Ethiopia datasets for SHAP explainability analysis.

import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

uk_df = pd.read_csv(
    "../data/processed/uk_processed.csv"
)

ethiopia_df = pd.read_csv(
    "../data/processed/ethiopia_processed.csv"
)

print("UK dataset shape:", uk_df.shape)
print("Ethiopia dataset shape:", ethiopia_df.shape)

print("\nUK target distribution:")
print(uk_df["severity_class"].value_counts())

print("\nEthiopia target distribution:")
print(ethiopia_df["severity_class"].value_counts())

In [ ]:
# Cell 2: Rebuild the final UK cost-sensitive CatBoost model with alpha=0.75 for SHAP explainability analysis.

shap_features = [
    "hour",
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common",
    "vehicle_count",
    "has_motorcycle",
    "has_heavy_vehicle",
    "has_public_transport",
    "has_two_wheeler"
]

shap_categorical = [
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common"
]

X_uk = uk_df[shap_features].copy()
y_uk = uk_df["severity_class"].copy()

X_train_uk, X_test_uk, y_train_uk, y_test_uk = train_test_split(
    X_uk,
    y_uk,
    test_size=0.20,
    random_state=42,
    stratify=y_uk
)

# Calculate balanced class weights using the UK training partition only
class_counts = y_train_uk.value_counts()

balanced_weights = {
    cls: len(y_train_uk) / (len(class_counts) * count)
    for cls, count in class_counts.items()
}

selected_alpha = 0.75

final_class_weights = {
    cls: weight ** selected_alpha
    for cls, weight in balanced_weights.items()
}

uk_shap_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=final_class_weights
)

uk_shap_model.fit(
    X_train_uk,
    y_train_uk,
    cat_features=shap_categorical
)

y_pred_uk = uk_shap_model.predict(X_test_uk).ravel()

print("UK SHAP model trained successfully.")
print("Selected alpha:", selected_alpha)

print("\nFinal class weights:")
for cls, weight in final_class_weights.items():
    print(f"{cls}: {weight:.4f}")

print("\nModel classes:")
print(uk_shap_model.classes_)

print("\nUK internal validation:")
print(f"Accuracy: {accuracy_score(y_test_uk, y_pred_uk):.4f}")
print(
    "Macro F1:",
    round(
        f1_score(
            y_test_uk,
            y_pred_uk,
            average="macro",
            zero_division=0
        ),
        4
    )
)
print(
    "Macro Recall:",
    round(
        recall_score(
            y_test_uk,
            y_pred_uk,
            average="macro",
            zero_division=0
        ),
        4
    )
)

In [ ]:
# Cell 3: Compute multiclass SHAP values using CatBoost's native SHAP implementation on a memory-efficient UK test sample.

from catboost import Pool
import numpy as np
import pandas as pd
import gc

gc.collect()

# Use a smaller representative sample to avoid memory-related kernel crashes
shap_sample = X_test_uk.sample(
    n=min(500, len(X_test_uk)),
    random_state=42
).copy()

shap_pool = Pool(
    shap_sample,
    cat_features=shap_categorical
)

# Native CatBoost SHAP computation
shap_values_native = uk_shap_model.get_feature_importance(
    shap_pool,
    type="ShapValues"
)

shap_values_native = np.asarray(shap_values_native)

print("Native CatBoost SHAP computation completed successfully.")

print("\nSHAP sample shape:")
print(shap_sample.shape)

print("\nModel classes:")
print(uk_shap_model.classes_)

print("\nNative SHAP array shape:")
print(shap_values_native.shape)

print("\nNumber of features:")
print(len(shap_features))

In [ ]:
# Cell 4: Calculate global SHAP feature importance across all three severity classes using native CatBoost SHAP values.

# Remove the final expected-value column
shap_feature_values = shap_values_native[:, :, :-1]

# Mean absolute SHAP across samples and classes
global_shap_importance = np.mean(
    np.abs(shap_feature_values),
    axis=(0, 1)
)

global_shap_df = pd.DataFrame({
    "Feature": shap_features,
    "Mean_Abs_SHAP": global_shap_importance
})

global_shap_df = (
    global_shap_df
    .sort_values("Mean_Abs_SHAP", ascending=False)
    .reset_index(drop=True)
)

global_shap_df["Importance_Percent"] = (
    global_shap_df["Mean_Abs_SHAP"]
    / global_shap_df["Mean_Abs_SHAP"].sum()
    * 100
).round(2)

global_shap_df["Rank"] = np.arange(
    1,
    len(global_shap_df) + 1
)

print("UK Final Model — Global SHAP Feature Importance")
print("=" * 75)

print(
    global_shap_df[
        [
            "Rank",
            "Feature",
            "Mean_Abs_SHAP",
            "Importance_Percent"
        ]
    ].round(4).to_string(index=False)
)

In [ ]:
# Cell 5: Calculate class-specific SHAP feature importance for Slight, Serious, and Fatal crash predictions.

class_names = list(uk_shap_model.classes_)

class_specific_shap_tables = {}

for class_idx, class_name in enumerate(class_names):

    # SHAP values for one class, excluding expected value
    class_shap_values = shap_feature_values[:, class_idx, :]

    class_importance = np.mean(
        np.abs(class_shap_values),
        axis=0
    )

    class_df = pd.DataFrame({
        "Feature": shap_features,
        "Mean_Abs_SHAP": class_importance
    })

    class_df = (
        class_df
        .sort_values(
            "Mean_Abs_SHAP",
            ascending=False
        )
        .reset_index(drop=True)
    )

    class_df["Importance_Percent"] = (
        class_df["Mean_Abs_SHAP"]
        / class_df["Mean_Abs_SHAP"].sum()
        * 100
    ).round(2)

    class_df["Rank"] = np.arange(
        1,
        len(class_df) + 1
    )

    class_specific_shap_tables[class_name] = class_df

    print("\n" + "=" * 80)
    print(f"SHAP Feature Importance — {class_name}")
    print("=" * 80)

    print(
        class_df[
            [
                "Rank",
                "Feature",
                "Mean_Abs_SHAP",
                "Importance_Percent"
            ]
        ].round(4).to_string(index=False)
    )

In [ ]:
# Cell 6: Build a class-specific SHAP comparison table for manuscript reporting.

shap_class_comparison = pd.DataFrame({
    "Feature": shap_features
})

for class_name in ["Slight", "Serious", "Fatal"]:

    temp = (
        class_specific_shap_tables[class_name]
        .set_index("Feature")
    )

    shap_class_comparison[
        f"{class_name}_MeanAbsSHAP"
    ] = shap_class_comparison["Feature"].map(
        temp["Mean_Abs_SHAP"]
    )

    shap_class_comparison[
        f"{class_name}_Importance_%"
    ] = shap_class_comparison["Feature"].map(
        temp["Importance_Percent"]
    )

    shap_class_comparison[
        f"{class_name}_Rank"
    ] = shap_class_comparison["Feature"].map(
        temp["Rank"]
    )


# Add global importance
global_temp = global_shap_df.set_index("Feature")

shap_class_comparison["Global_Importance_%"] = (
    shap_class_comparison["Feature"].map(
        global_temp["Importance_Percent"]
    )
)

shap_class_comparison["Global_Rank"] = (
    shap_class_comparison["Feature"].map(
        global_temp["Rank"]
    )
)

shap_class_comparison = (
    shap_class_comparison
    .sort_values("Global_Rank")
    .reset_index(drop=True)
)

print(
    "UK Cost-Sensitive CatBoost — "
    "Class-Specific SHAP Comparison"
)

print("=" * 120)

print(
    shap_class_comparison.round(4).to_string(
        index=False
    )
)

In [ ]:
from pathlib import Path

table_dir = Path("../outputs/tables")
table_dir.mkdir(parents=True, exist_ok=True)

shap_class_comparison.to_csv(
    table_dir /
    "Table_UK_Class_Specific_SHAP_Comparison.csv",
    index=False
)

global_shap_df.to_csv(
    table_dir /
    "Table_UK_Global_SHAP_Importance.csv",
    index=False
)

print("Saved:")
print("1. Table_UK_Class_Specific_SHAP_Comparison.csv")
print("2. Table_UK_Global_SHAP_Importance.csv")

In [ ]:
# Cell 8: Analyze the direction of SHAP effects for the Fatal class.
# Positive mean SHAP -> pushes model output toward Fatal.
# Negative mean SHAP -> pushes model output away from Fatal.

fatal_class_idx = list(uk_shap_model.classes_).index("Fatal")

fatal_shap = shap_feature_values[:, fatal_class_idx, :]

fatal_shap_df = pd.DataFrame(
    fatal_shap,
    columns=shap_features,
    index=shap_sample.index
)

categorical_features_for_direction = [
    "junction_common",
    "weather_common",
    "light_common",
    "surface_common",
    "day_common"
]

fatal_category_results = []

for feature in categorical_features_for_direction:

    temp = pd.DataFrame({
        "Category": shap_sample[feature].astype(str),
        "SHAP": fatal_shap_df[feature]
    })

    summary = (
        temp
        .groupby("Category")
        .agg(
            Mean_SHAP=("SHAP", "mean"),
            Mean_Abs_SHAP=("SHAP", lambda x: np.mean(np.abs(x))),
            N=("SHAP", "size")
        )
        .reset_index()
    )

    summary["Feature"] = feature

    summary["Direction"] = np.where(
        summary["Mean_SHAP"] > 0,
        "Toward Fatal",
        np.where(
            summary["Mean_SHAP"] < 0,
            "Away from Fatal",
            "Neutral"
        )
    )

    fatal_category_results.append(summary)


fatal_category_effects_df = pd.concat(
    fatal_category_results,
    ignore_index=True
)

fatal_category_effects_df = fatal_category_effects_df[
    [
        "Feature",
        "Category",
        "N",
        "Mean_SHAP",
        "Mean_Abs_SHAP",
        "Direction"
    ]
]

print(
    "UK Final Model — Direction of Categorical "
    "SHAP Effects for Fatal Prediction"
)

print("=" * 100)

for feature in categorical_features_for_direction:

    print("\n" + feature)
    print("-" * 80)

    display_df = (
        fatal_category_effects_df[
            fatal_category_effects_df["Feature"] == feature
        ]
        .sort_values(
            "Mean_SHAP",
            ascending=False
        )
    )

    print(
        display_df[
            [
                "Category",
                "N",
                "Mean_SHAP",
                "Mean_Abs_SHAP",
                "Direction"
            ]
        ].round(4).to_string(index=False)
    )

In [ ]:

# Cell 9: Analyze signed SHAP effects of numerical and binary predictors
# on Fatal-class model output.

fatal_numeric_features = [
    "vehicle_count",
    "hour",
    "has_motorcycle",
    "has_two_wheeler",
    "has_heavy_vehicle",
    "has_public_transport"
]

# ---------------------------------------------------------
# Binary features
# ---------------------------------------------------------

binary_features = [
    "has_motorcycle",
    "has_two_wheeler",
    "has_heavy_vehicle",
    "has_public_transport"
]

binary_results = []

for feature in binary_features:

    temp = pd.DataFrame({
        "Value": shap_sample[feature],
        "SHAP": fatal_shap_df[feature]
    })

    summary = (
        temp
        .groupby("Value")
        .agg(
            N=("SHAP", "size"),
            Mean_SHAP=("SHAP", "mean"),
            Mean_Abs_SHAP=(
                "SHAP",
                lambda x: np.mean(np.abs(x))
            )
        )
        .reset_index()
    )

    summary["Feature"] = feature

    summary["Direction"] = np.where(
        summary["Mean_SHAP"] > 0,
        "Toward Fatal",
        np.where(
            summary["Mean_SHAP"] < 0,
            "Away from Fatal",
            "Neutral"
        )
    )

    binary_results.append(summary)


fatal_binary_effects_df = pd.concat(
    binary_results,
    ignore_index=True
)

fatal_binary_effects_df = fatal_binary_effects_df[
    [
        "Feature",
        "Value",
        "N",
        "Mean_SHAP",
        "Mean_Abs_SHAP",
        "Direction"
    ]
]


print("Binary Feature SHAP Effects — Fatal")
print("=" * 85)

for feature in binary_features:

    print("\n" + feature)
    print("-" * 65)

    display_df = fatal_binary_effects_df[
        fatal_binary_effects_df["Feature"] == feature
    ]

    print(
        display_df.round(4).to_string(index=False)
    )


# ---------------------------------------------------------
# Vehicle count
# ---------------------------------------------------------

vehicle_count_effects = pd.DataFrame({
    "vehicle_count": shap_sample["vehicle_count"],
    "Fatal_SHAP": fatal_shap_df["vehicle_count"]
})

vehicle_count_summary = (
    vehicle_count_effects
    .groupby("vehicle_count")
    .agg(
        N=("Fatal_SHAP", "size"),
        Mean_SHAP=("Fatal_SHAP", "mean"),
        Mean_Abs_SHAP=(
            "Fatal_SHAP",
            lambda x: np.mean(np.abs(x))
        )
    )
    .reset_index()
)

vehicle_count_summary["Direction"] = np.where(
    vehicle_count_summary["Mean_SHAP"] > 0,
    "Toward Fatal",
    np.where(
        vehicle_count_summary["Mean_SHAP"] < 0,
        "Away from Fatal",
        "Neutral"
    )
)

print("\n\nVehicle Count SHAP Effects — Fatal")
print("=" * 85)

print(
    vehicle_count_summary
    .round(4)
    .to_string(index=False)
)


# ---------------------------------------------------------
# Hour
# ---------------------------------------------------------

hour_effects = pd.DataFrame({
    "hour": shap_sample["hour"],
    "Fatal_SHAP": fatal_shap_df["hour"]
})

hour_summary = (
    hour_effects
    .groupby("hour")
    .agg(
        N=("Fatal_SHAP", "size"),
        Mean_SHAP=("Fatal_SHAP", "mean"),
        Mean_Abs_SHAP=(
            "Fatal_SHAP",
            lambda x: np.mean(np.abs(x))
        )
    )
    .reset_index()
)

hour_summary["Direction"] = np.where(
    hour_summary["Mean_SHAP"] > 0,
    "Toward Fatal",
    np.where(
        hour_summary["Mean_SHAP"] < 0,
        "Away from Fatal",
        "Neutral"
    )
)

print("\n\nHour SHAP Effects — Fatal")
print("=" * 85)

print(
    hour_summary
    .round(4)
    .to_string(index=False)
)

In [ ]:
# Cell 10: Create a publication-ready SHAP figure for Fatal-class interpretation.

import matplotlib.pyplot as plt
from pathlib import Path

figure_dir = Path("../outputs/figures")
figure_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------
# Data preparation
# -------------------------------

fatal_importance_plot = (
    class_specific_shap_tables["Fatal"]
    .sort_values("Mean_Abs_SHAP", ascending=True)
)

junction_plot = (
    fatal_category_effects_df[
        fatal_category_effects_df["Feature"] == "junction_common"
    ]
    .sort_values("Mean_SHAP", ascending=True)
)

vehicle_plot = (
    vehicle_count_summary
    .sort_values("vehicle_count")
)

# -------------------------------
# Create figure
# -------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(17, 6)
)

# Panel A — Fatal feature importance
axes[0].barh(
    fatal_importance_plot["Feature"],
    fatal_importance_plot["Mean_Abs_SHAP"]
)

axes[0].set_xlabel("Mean |SHAP value|")
axes[0].set_title(
    "A. Fatal-Class Feature Importance"
)

# Panel B — Junction signed effects
axes[1].barh(
    junction_plot["Category"],
    junction_plot["Mean_SHAP"]
)

axes[1].axvline(
    0,
    linewidth=1
)

axes[1].set_xlabel(
    "Mean SHAP value for Fatal class"
)

axes[1].set_title(
    "B. Junction Effects"
)

# Add sample sizes
for i, (_, row) in enumerate(
    junction_plot.iterrows()
):

    value = row["Mean_SHAP"]

    if value >= 0:
        x_position = value + 0.015
        alignment = "left"
    else:
        x_position = value - 0.015
        alignment = "right"

    axes[1].text(
        x_position,
        i,
        f"n={int(row['N'])}",
        va="center",
        ha=alignment,
        fontsize=8
    )

# Panel C — Vehicle count
axes[2].plot(
    vehicle_plot["vehicle_count"],
    vehicle_plot["Mean_SHAP"],
    marker="o"
)

axes[2].axhline(
    0,
    linewidth=1
)

axes[2].set_xlabel(
    "Number of vehicles"
)

axes[2].set_ylabel(
    "Mean SHAP value for Fatal class"
)

axes[2].set_title(
    "C. Vehicle-Count Effect"
)

for _, row in vehicle_plot.iterrows():

    axes[2].annotate(
        f"n={int(row['N'])}",
        (
            row["vehicle_count"],
            row["Mean_SHAP"]
        ),
        xytext=(4, 5),
        textcoords="offset points",
        fontsize=8
    )

fig.suptitle(
    "SHAP Interpretation of Fatal Crash Predictions — UK Model",
    fontsize=15
)

plt.tight_layout()

output_path = (
    figure_dir /
    "Figure_UK_Fatal_SHAP_Interpretation.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(output_path)

In [ ]:
# Cell 11: Evaluate stability of global SHAP feature importance
# across five independent 500-case samples from the UK test set.

from catboost import Pool
import numpy as np
import pandas as pd
import gc

shap_seeds = [11, 22, 33, 44, 55]

stability_results = []

for seed in shap_seeds:

    print(f"Processing SHAP sample — seed {seed}...")

    sample = X_test_uk.sample(
        n=min(500, len(X_test_uk)),
        random_state=seed
    ).copy()

    sample_pool = Pool(
        sample,
        cat_features=shap_categorical
    )

    values = uk_shap_model.get_feature_importance(
        sample_pool,
        type="ShapValues"
    )

    values = np.asarray(values)

    # Remove expected-value column
    feature_values = values[:, :, :-1]

    # Global mean absolute SHAP
    importance = np.mean(
        np.abs(feature_values),
        axis=(0, 1)
    )

    total_importance = importance.sum()

    for feature, value in zip(
        shap_features,
        importance
    ):

        stability_results.append({
            "Seed": seed,
            "Feature": feature,
            "Mean_Abs_SHAP": value,
            "Importance_Percent":
                value / total_importance * 100
        })

    del sample_pool
    del values
    del feature_values

    gc.collect()


shap_stability_long = pd.DataFrame(
    stability_results
)

# --------------------------------------------------
# Mean and SD across five independent samples
# --------------------------------------------------

shap_stability_summary = (
    shap_stability_long
    .groupby("Feature")
    .agg(
        Mean_Importance_Percent=(
            "Importance_Percent",
            "mean"
        ),
        SD_Importance_Percent=(
            "Importance_Percent",
            "std"
        ),
        Min_Importance_Percent=(
            "Importance_Percent",
            "min"
        ),
        Max_Importance_Percent=(
            "Importance_Percent",
            "max"
        )
    )
    .reset_index()
)

shap_stability_summary = (
    shap_stability_summary
    .sort_values(
        "Mean_Importance_Percent",
        ascending=False
    )
    .reset_index(drop=True)
)

shap_stability_summary["Mean_Rank"] = (
    shap_stability_long
    .assign(
        Rank=lambda x:
            x.groupby("Seed")[
                "Importance_Percent"
            ]
            .rank(
                ascending=False,
                method="average"
            )
    )
    .groupby("Feature")["Rank"]
    .mean()
    .reindex(
        shap_stability_summary["Feature"]
    )
    .values
)

print(
    "\nUK Global SHAP Stability — "
    "5 Independent Samples"
)

print("=" * 95)

print(
    shap_stability_summary
    .round(3)
    .to_string(index=False)
)

In [ ]:
shap_stability_long.to_csv(
    table_dir /
    "Table_UK_SHAP_Stability_All_Samples.csv",
    index=False
)

shap_stability_summary.to_csv(
    table_dir /
    "Table_UK_SHAP_Stability_Summary.csv",
    index=False
)

print("Saved:")
print("1. Table_UK_SHAP_Stability_All_Samples.csv")
print("2. Table_UK_SHAP_Stability_Summary.csv")

In [ ]:
# Cell 13: Rebuild the weighted Ethiopia Model B with accident/environment and driver features for SHAP analysis.

ethiopia_base_features = [
    "hour",
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common",
    "vehicle_category",
    "vehicle_count",
    "has_motorcycle",
    "has_heavy_vehicle",
    "has_public_transport",
    "has_two_wheeler",
    "Type_of_collision",
    "Number_of_casualties",
    "Road_surface_type",
    "Road_allignment",
    "Area_accident_occured",
    "Vehicle_movement",
    "Cause_of_accident"
]

ethiopia_driver_features = [
    "Age_band_of_driver",
    "Drivers_gender",
    "Educational_level",
    "Driving_experience",
    "Vehicle_driver_relation"
]

ethiopia_features_B = (
    ethiopia_base_features
    + ethiopia_driver_features
)

eth_shap_df = ethiopia_df[
    ethiopia_features_B + ["severity_class"]
].copy()

# Preserve missing categorical information
for col in ethiopia_features_B:
    if eth_shap_df[col].dtype == "object":
        eth_shap_df[col] = (
            eth_shap_df[col]
            .fillna("Missing")
            .astype(str)
        )

X_eth = eth_shap_df[ethiopia_features_B].copy()
y_eth = eth_shap_df["severity_class"].copy()

X_train_eth, X_test_eth, y_train_eth, y_test_eth = train_test_split(
    X_eth,
    y_eth,
    test_size=0.20,
    random_state=42,
    stratify=y_eth
)

eth_categorical = [
    col for col in ethiopia_features_B
    if X_train_eth[col].dtype == "object"
]

# Balanced weights from training data only
eth_counts = y_train_eth.value_counts()

eth_weights = {
    cls: len(y_train_eth) / (len(eth_counts) * count)
    for cls, count in eth_counts.items()
}

ethiopia_shap_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=eth_weights
)

ethiopia_shap_model.fit(
    X_train_eth,
    y_train_eth,
    cat_features=eth_categorical
)

y_pred_eth = ethiopia_shap_model.predict(
    X_test_eth
).ravel()

print("Ethiopia SHAP model trained successfully.")

print("\nModel classes:")
print(ethiopia_shap_model.classes_)

print("\nClass weights:")
for cls, weight in eth_weights.items():
    print(f"{cls}: {weight:.4f}")

print("\nInternal validation:")
print(
    "Accuracy:",
    round(
        accuracy_score(y_test_eth, y_pred_eth),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_test_eth,
            y_pred_eth,
            average="macro",
            zero_division=0
        ),
        4
    )
)

In [ ]:
# Cell 14: Compute native multiclass SHAP values for a representative Ethiopia test sample using the weighted Model B.

from catboost import Pool
import numpy as np
import pandas as pd
import gc

gc.collect()

eth_shap_sample = X_test_eth.sample(
    n=min(500, len(X_test_eth)),
    random_state=42
).copy()

eth_shap_pool = Pool(
    eth_shap_sample,
    cat_features=eth_categorical
)

eth_shap_values_native = (
    ethiopia_shap_model.get_feature_importance(
        eth_shap_pool,
        type="ShapValues"
    )
)

eth_shap_values_native = np.asarray(
    eth_shap_values_native
)

print("Ethiopia native SHAP computation completed successfully.")

print("\nSHAP sample shape:")
print(eth_shap_sample.shape)

print("\nModel classes:")
print(ethiopia_shap_model.classes_)

print("\nNative SHAP array shape:")
print(eth_shap_values_native.shape)

print("\nNumber of model features:")
print(len(ethiopia_features_B))

In [ ]:
# Cell 15: Calculate global SHAP importance for Ethiopia Model B
# and quantify the contribution of driver vs accident/environment features.

# Remove expected-value column
eth_shap_feature_values = eth_shap_values_native[:, :, :-1]

# Mean absolute SHAP across all samples and all three classes
eth_global_shap = np.mean(
    np.abs(eth_shap_feature_values),
    axis=(0, 1)
)

eth_global_shap_df = pd.DataFrame({
    "Feature": ethiopia_features_B,
    "Mean_Abs_SHAP": eth_global_shap
})

# Feature groups
eth_global_shap_df["Feature_Group"] = np.where(
    eth_global_shap_df["Feature"].isin(
        ethiopia_driver_features
    ),
    "Driver",
    "Accident/Environment"
)

# Sort by importance
eth_global_shap_df = (
    eth_global_shap_df
    .sort_values(
        "Mean_Abs_SHAP",
        ascending=False
    )
    .reset_index(drop=True)
)

# Percentage contribution
eth_global_shap_df["Importance_Percent"] = (
    eth_global_shap_df["Mean_Abs_SHAP"]
    / eth_global_shap_df["Mean_Abs_SHAP"].sum()
    * 100
)

eth_global_shap_df["Rank"] = np.arange(
    1,
    len(eth_global_shap_df) + 1
)

print(
    "Ethiopia Model B — Global SHAP Feature Importance"
)

print("=" * 100)

print(
    eth_global_shap_df[
        [
            "Rank",
            "Feature",
            "Feature_Group",
            "Mean_Abs_SHAP",
            "Importance_Percent"
        ]
    ].round(4).to_string(index=False)
)


# ---------------------------------------------------------
# Group-level contribution
# ---------------------------------------------------------

eth_shap_group_summary = (
    eth_global_shap_df
    .groupby("Feature_Group")
    .agg(
        Total_Mean_Abs_SHAP=(
            "Mean_Abs_SHAP",
            "sum"
        ),
        Total_Importance_Percent=(
            "Importance_Percent",
            "sum"
        ),
        Number_of_Features=(
            "Feature",
            "count"
        )
    )
    .reset_index()
)

print(
    "\n\nSHAP Contribution by Feature Group"
)

print("=" * 75)

print(
    eth_shap_group_summary
    .round(4)
    .to_string(index=False)
)


# ---------------------------------------------------------
# Driver features only
# ---------------------------------------------------------

eth_driver_shap_df = (
    eth_global_shap_df[
        eth_global_shap_df[
            "Feature_Group"
        ] == "Driver"
    ]
    .copy()
)

print(
    "\n\nDriver Features — SHAP Importance"
)

print("=" * 75)

print(
    eth_driver_shap_df[
        [
            "Rank",
            "Feature",
            "Mean_Abs_SHAP",
            "Importance_Percent"
        ]
    ].round(4).to_string(index=False)
)

In [ ]:
# Cell 16: Class-specific SHAP analysis for Ethiopia Model B
# Includes total Driver vs Accident/Environment contribution
# separately for Slight, Serious, and Fatal.

eth_class_names = list(
    ethiopia_shap_model.classes_
)

eth_class_specific_tables = []
eth_class_group_tables = []

for class_idx, class_name in enumerate(
    eth_class_names
):

    # SHAP values for this class
    class_values = (
        eth_shap_feature_values[
            :, class_idx, :
        ]
    )

    # Mean absolute SHAP
    class_importance = np.mean(
        np.abs(class_values),
        axis=0
    )

    temp_df = pd.DataFrame({
        "Feature": ethiopia_features_B,
        "Mean_Abs_SHAP": class_importance
    })

    temp_df["Feature_Group"] = np.where(
        temp_df["Feature"].isin(
            ethiopia_driver_features
        ),
        "Driver",
        "Accident/Environment"
    )

    temp_df["Importance_Percent"] = (
        temp_df["Mean_Abs_SHAP"]
        / temp_df["Mean_Abs_SHAP"].sum()
        * 100
    )

    temp_df = (
        temp_df
        .sort_values(
            "Mean_Abs_SHAP",
            ascending=False
        )
        .reset_index(drop=True)
    )

    temp_df["Rank"] = np.arange(
        1,
        len(temp_df) + 1
    )

    temp_df["Class"] = class_name

    eth_class_specific_tables.append(
        temp_df
    )

    # Group-level contribution
    group_df = (
        temp_df
        .groupby("Feature_Group")
        .agg(
            Total_Mean_Abs_SHAP=(
                "Mean_Abs_SHAP",
                "sum"
            ),
            Importance_Percent=(
                "Importance_Percent",
                "sum"
            )
        )
        .reset_index()
    )

    group_df["Class"] = class_name

    eth_class_group_tables.append(
        group_df
    )


eth_class_shap_df = pd.concat(
    eth_class_specific_tables,
    ignore_index=True
)

eth_class_group_df = pd.concat(
    eth_class_group_tables,
    ignore_index=True
)


# ============================================
# Print feature rankings for each class
# ============================================

for class_name in eth_class_names:

    print("\n" + "=" * 100)
    print(
        f"Ethiopia SHAP Feature Importance — "
        f"{class_name}"
    )
    print("=" * 100)

    display_df = (
        eth_class_shap_df[
            eth_class_shap_df["Class"]
            == class_name
        ]
        .sort_values("Rank")
    )

    print(
        display_df[
            [
                "Rank",
                "Feature",
                "Feature_Group",
                "Mean_Abs_SHAP",
                "Importance_Percent"
            ]
        ]
        .round(4)
        .to_string(index=False)
    )


# ============================================
# Driver vs Accident/Environment comparison
# ============================================

print("\n\n" + "=" * 85)
print(
    "Driver vs Accident/Environment SHAP "
    "Contribution by Severity Class"
)
print("=" * 85)

group_pivot = (
    eth_class_group_df
    .pivot(
        index="Class",
        columns="Feature_Group",
        values="Importance_Percent"
    )
    .reset_index()
)

group_pivot = group_pivot[
    [
        "Class",
        "Driver",
        "Accident/Environment"
    ]
]

print(
    group_pivot
    .round(2)
    .to_string(index=False)
)

In [ ]:
# Cell 17: Direction of driver-feature SHAP effects
# specifically for Fatal crash prediction in Ethiopia.

fatal_idx_eth = list(
    ethiopia_shap_model.classes_
).index("Fatal")

eth_fatal_shap = (
    eth_shap_feature_values[
        :, fatal_idx_eth, :
    ]
)

eth_fatal_shap_df = pd.DataFrame(
    eth_fatal_shap,
    columns=ethiopia_features_B,
    index=eth_shap_sample.index
)

driver_direction_results = []

for feature in ethiopia_driver_features:

    temp = pd.DataFrame({
        "Category":
            eth_shap_sample[feature].astype(str),
        "SHAP":
            eth_fatal_shap_df[feature]
    })

    summary = (
        temp
        .groupby("Category")
        .agg(
            N=("SHAP", "size"),
            Mean_SHAP=("SHAP", "mean"),
            Median_SHAP=("SHAP", "median"),
            Mean_Abs_SHAP=(
                "SHAP",
                lambda x: np.mean(np.abs(x))
            )
        )
        .reset_index()
    )

    summary["Feature"] = feature

    summary["Direction"] = np.where(
        summary["Mean_SHAP"] > 0,
        "Toward Fatal",
        np.where(
            summary["Mean_SHAP"] < 0,
            "Away from Fatal",
            "Neutral"
        )
    )

    driver_direction_results.append(
        summary
    )


eth_driver_fatal_direction_df = pd.concat(
    driver_direction_results,
    ignore_index=True
)

eth_driver_fatal_direction_df = (
    eth_driver_fatal_direction_df[
        [
            "Feature",
            "Category",
            "N",
            "Mean_SHAP",
            "Median_SHAP",
            "Mean_Abs_SHAP",
            "Direction"
        ]
    ]
)


print(
    "Ethiopia — Driver Feature Effects "
    "on Fatal Prediction"
)

print("=" * 100)

for feature in ethiopia_driver_features:

    print("\n" + feature)
    print("-" * 90)

    display_df = (
        eth_driver_fatal_direction_df[
            eth_driver_fatal_direction_df[
                "Feature"
            ] == feature
        ]
        .sort_values(
            "Mean_SHAP",
            ascending=False
        )
    )

    print(
        display_df
        .round(4)
        .to_string(index=False)
    )

In [ ]:
# Cell 18: Publication-ready figure showing signed SHAP effects
# of driver age and driving experience on Fatal predictions.

import matplotlib.pyplot as plt
from pathlib import Path

figure_dir = Path("../outputs/figures")
figure_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------
# Age
# -------------------------------------------------------

age_plot = (
    eth_driver_fatal_direction_df[
        eth_driver_fatal_direction_df["Feature"]
        == "Age_band_of_driver"
    ]
    .copy()
)

age_order = [
    "Under 18",
    "18-30",
    "31-50",
    "Over 51",
    "Unknown"
]

age_plot["Category"] = pd.Categorical(
    age_plot["Category"],
    categories=age_order,
    ordered=True
)

age_plot = age_plot.sort_values("Category")


# -------------------------------------------------------
# Driving experience
# -------------------------------------------------------

experience_plot = (
    eth_driver_fatal_direction_df[
        eth_driver_fatal_direction_df["Feature"]
        == "Driving_experience"
    ]
    .copy()
)

experience_order = [
    "No Licence",
    "Below 1yr",
    "1-2yr",
    "2-5yr",
    "5-10yr",
    "Above 10yr",
    "Missing",
    "Unknown"
]

experience_plot["Category"] = pd.Categorical(
    experience_plot["Category"],
    categories=experience_order,
    ordered=True
)

experience_plot = experience_plot.sort_values("Category")


# -------------------------------------------------------
# Figure
# -------------------------------------------------------

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6)
)

# Age panel
axes[0].barh(
    age_plot["Category"].astype(str),
    age_plot["Mean_SHAP"]
)

axes[0].axvline(
    0,
    linewidth=1
)

axes[0].set_xlabel(
    "Mean SHAP value for Fatal class"
)

axes[0].set_title(
    "A. Driver Age"
)

for i, (_, row) in enumerate(age_plot.iterrows()):

    value = row["Mean_SHAP"]

    axes[0].text(
        value + (0.008 if value >= 0 else -0.008),
        i,
        f"n={int(row['N'])}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=8
    )


# Driving-experience panel
axes[1].barh(
    experience_plot["Category"].astype(str),
    experience_plot["Mean_SHAP"]
)

axes[1].axvline(
    0,
    linewidth=1
)

axes[1].set_xlabel(
    "Mean SHAP value for Fatal class"
)

axes[1].set_title(
    "B. Driving Experience"
)

for i, (_, row) in enumerate(
    experience_plot.iterrows()
):

    value = row["Mean_SHAP"]

    axes[1].text(
        value + (0.008 if value >= 0 else -0.008),
        i,
        f"n={int(row['N'])}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=8
    )


fig.suptitle(
    "Driver-Related SHAP Effects on Fatal Crash Predictions — Ethiopia",
    fontsize=14
)

plt.tight_layout()

output_path = (
    figure_dir /
    "Figure_Ethiopia_Driver_Fatal_SHAP.png"
)

plt.savefig(
    output_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(output_path)

In [ ]:
# Cell 19: SHAP stability analysis for Ethiopia Model B
# 5 independent samples × 500 observations

from catboost import Pool
import numpy as np
import pandas as pd
import gc

eth_shap_seeds = [11, 22, 33, 44, 55]

eth_stability_results = []
eth_group_stability_results = []

for seed in eth_shap_seeds:

    print(f"Processing Ethiopia SHAP sample — seed {seed}...")

    sample = X_test_eth.sample(
        n=min(500, len(X_test_eth)),
        random_state=seed
    ).copy()

    sample_pool = Pool(
        sample,
        cat_features=eth_categorical
    )

    values = ethiopia_shap_model.get_feature_importance(
        sample_pool,
        type="ShapValues"
    )

    values = np.asarray(values)

    # Remove expected-value column
    feature_values = values[:, :, :-1]

    # ------------------------------------------
    # Global SHAP importance
    # ------------------------------------------

    importance = np.mean(
        np.abs(feature_values),
        axis=(0, 1)
    )

    total_importance = importance.sum()

    sample_df = pd.DataFrame({
        "Feature": ethiopia_features_B,
        "Mean_Abs_SHAP": importance
    })

    sample_df["Importance_Percent"] = (
        sample_df["Mean_Abs_SHAP"]
        / total_importance
        * 100
    )

    sample_df["Feature_Group"] = np.where(
        sample_df["Feature"].isin(
            ethiopia_driver_features
        ),
        "Driver",
        "Accident/Environment"
    )

    sample_df["Rank"] = (
        sample_df["Importance_Percent"]
        .rank(
            ascending=False,
            method="average"
        )
    )

    sample_df["Seed"] = seed

    eth_stability_results.append(
        sample_df
    )

    # ------------------------------------------
    # Group-level contribution
    # ------------------------------------------

    group_df = (
        sample_df
        .groupby("Feature_Group")
        ["Importance_Percent"]
        .sum()
        .reset_index()
    )

    group_df["Seed"] = seed

    eth_group_stability_results.append(
        group_df
    )

    del sample_pool
    del values
    del feature_values
    del sample_df

    gc.collect()


# ==========================================================
# Combine all five runs
# ==========================================================

eth_shap_stability_long = pd.concat(
    eth_stability_results,
    ignore_index=True
)

eth_group_stability_long = pd.concat(
    eth_group_stability_results,
    ignore_index=True
)


# ==========================================================
# Feature-level stability
# ==========================================================

eth_shap_stability_summary = (
    eth_shap_stability_long
    .groupby("Feature")
    .agg(
        Mean_Importance_Percent=(
            "Importance_Percent",
            "mean"
        ),
        SD_Importance_Percent=(
            "Importance_Percent",
            "std"
        ),
        Min_Importance_Percent=(
            "Importance_Percent",
            "min"
        ),
        Max_Importance_Percent=(
            "Importance_Percent",
            "max"
        ),
        Mean_Rank=(
            "Rank",
            "mean"
        ),
        SD_Rank=(
            "Rank",
            "std"
        )
    )
    .reset_index()
)

eth_shap_stability_summary = (
    eth_shap_stability_summary
    .sort_values(
        "Mean_Importance_Percent",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\nEthiopia Global SHAP Stability — "
    "5 Independent Samples"
)

print("=" * 110)

print(
    eth_shap_stability_summary
    .round(3)
    .to_string(index=False)
)


# ==========================================================
# Driver vs environment stability
# ==========================================================

eth_group_stability_summary = (
    eth_group_stability_long
    .groupby("Feature_Group")
    .agg(
        Mean_Percent=(
            "Importance_Percent",
            "mean"
        ),
        SD_Percent=(
            "Importance_Percent",
            "std"
        ),
        Min_Percent=(
            "Importance_Percent",
            "min"
        ),
        Max_Percent=(
            "Importance_Percent",
            "max"
        )
    )
    .reset_index()
)

print(
    "\n\nDriver vs Accident/Environment "
    "SHAP Stability"
)

print("=" * 85)

print(
    eth_group_stability_summary
    .round(3)
    .to_string(index=False)
)

In [ ]:
eth_shap_stability_long.to_csv(
    table_dir /
    "Table_Ethiopia_SHAP_Stability_All_Samples.csv",
    index=False
)

eth_shap_stability_summary.to_csv(
    table_dir /
    "Table_Ethiopia_SHAP_Stability_Summary.csv",
    index=False
)

eth_group_stability_long.to_csv(
    table_dir /
    "Table_Ethiopia_SHAP_Group_Stability_All_Samples.csv",
    index=False
)

eth_group_stability_summary.to_csv(
    table_dir /
    "Table_Ethiopia_SHAP_Group_Stability_Summary.csv",
    index=False
)

print("Saved:")
print("1. Table_Ethiopia_SHAP_Stability_All_Samples.csv")
print("2. Table_Ethiopia_SHAP_Stability_Summary.csv")
print("3. Table_Ethiopia_SHAP_Group_Stability_All_Samples.csv")
print("4. Table_Ethiopia_SHAP_Group_Stability_Summary.csv")

In [ ]:
# Cell 21: Save final UK and Ethiopia SHAP results and create a manuscript-ready XAI summary table.

from pathlib import Path
import pandas as pd

table_dir = Path("../outputs/tables")
table_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Save Ethiopia global SHAP
# --------------------------------------------------

eth_global_shap_df.to_csv(
    table_dir / "Table_Ethiopia_Global_SHAP_Importance.csv",
    index=False
)

# --------------------------------------------------
# Save Ethiopia class-specific SHAP
# --------------------------------------------------

eth_class_shap_df.to_csv(
    table_dir / "Table_Ethiopia_Class_Specific_SHAP.csv",
    index=False
)

# --------------------------------------------------
# Save Ethiopia driver Fatal-direction analysis
# --------------------------------------------------

eth_driver_fatal_direction_df.to_csv(
    table_dir / "Table_Ethiopia_Driver_Fatal_SHAP_Direction.csv",
    index=False
)

# --------------------------------------------------
# Save SHAP group contribution by severity class
# --------------------------------------------------

group_pivot.to_csv(
    table_dir / "Table_Ethiopia_Driver_vs_Environment_SHAP_by_Class.csv",
    index=False
)

# --------------------------------------------------
# Create concise manuscript summary
# --------------------------------------------------

xai_summary = pd.DataFrame({
    "Finding": [
        "UK most important global SHAP feature",
        "UK second most important global SHAP feature",
        "UK third most important global SHAP feature",
        "UK most important Fatal SHAP feature",
        "Ethiopia driver SHAP contribution",
        "Ethiopia accident/environment SHAP contribution",
        "Ethiopia driver contribution to Fatal",
        "Ethiopia driver contribution to Serious",
        "Ethiopia driver contribution to Slight",
        "Ethiopia top driver feature",
        "Ethiopia second driver feature",
        "Driver-feature incremental Macro-F1 gain in repeated CV"
    ],
    "Result": [
        "junction_common",
        "vehicle_count",
        "weather_common",
        "junction_common",
        "19.46% ± 0.50%",
        "80.54% ± 0.50%",
        "21.51%",
        "17.66%",
        "20.49%",
        "Age_band_of_driver",
        "Driving_experience",
        "+0.0030"
    ]
})

xai_summary.to_csv(
    table_dir / "Table_Final_XAI_Findings_Summary.csv",
    index=False
)

print("Saved final XAI tables:")
print("1. Table_Ethiopia_Global_SHAP_Importance.csv")
print("2. Table_Ethiopia_Class_Specific_SHAP.csv")
print("3. Table_Ethiopia_Driver_Fatal_SHAP_Direction.csv")
print("4. Table_Ethiopia_Driver_vs_Environment_SHAP_by_Class.csv")
print("5. Table_Final_XAI_Findings_Summary.csv")